# Ajuste fino de BERT para reconocimiento de entidades nombradas (NER)

**Curso:** Procesamiento de Lenguaje Natural · Unidad 5 — Etiquetado de secuencias  
**Institución:** Universidad Tecnológica de Bolívar  
**Referencia:** Jurafsky, D. & Martin, J. H. (2025). *Speech and Language Processing* (3.ª ed.), Cap. 17 — Sequence Labeling for Parts of Speech and Named Entities  
**Compatibilidad con Python:** 3.10 · 3.11 · 3.12 · 3.13

## Contexto y Propósito

Este notebook entrena un modelo de **reconocimiento de entidades nombradas (NER)**: dado un texto, marca qué palabras son personas, lugares, organizaciones, entidades geopolíticas o expresiones de tiempo. Parte de **BERT** preentrenado y lo ajusta (*fine-tuning*) como clasificador de tokens con un corpus en inglés anotado en formato BIO. El punto delicado es que BERT divide las palabras en subpalabras, así que hay que alinear cada etiqueta con la primera subpalabra de su palabra. Se evalúa con F1 por entidad (`seqeval`), se prueba con frases nuevas y se guarda el modelo.

**Objetivo:** aprender el flujo completo de *fine-tuning* de un Transformer para una tarea de etiquetado de secuencias, desde la preparación de los datos y la alineación de etiquetas hasta la evaluación por entidad y el guardado del modelo para reutilizarlo.

## Ejemplos de Aplicación en el Mundo Real

Cada caso muestra, paso a paso, cómo se aplica la técnica de este notebook a un problema real.

### Finanzas y servicio al cliente — Entidades en noticias financieras

- **Problema:** seguir en tiempo real las noticias sobre los emisores de un portafolio.
- **Entrada:** texto de noticias.
- **Cómo se aplica NER con BERT:**
  1. BERT etiqueta cada subpalabra; la etiqueta de una palabra es la de su primera subpalabra (`Bancolombia` → `Banco` `##lombia`).
  2. Se ajusta con oraciones anotadas en formato BIO (ORG, PER, TIM…).
  3. Se evalúa con F1 por entidad (`seqeval`), no por token.
- **Salida:** empresas, personas y fechas mencionadas en cada noticia.
- **Decisión:** alerta cuando aparece un emisor del portafolio.

### Salud — Entidades clínicas

- **Problema:** extraer medicamentos, dosis y diagnósticos de notas clínicas.
- **Entrada:** notas clínicas anonimizadas.
- **Cómo se aplica NER con BERT:**
  1. Se definen etiquetas propias en BIO: MED, DOSIS, DX.
  2. Se alinean las etiquetas con las subpalabras de términos largos (`levotiroxina`).
  3. Se evalúa con F1 por entidad y se revisan los errores con personal clínico.
- **Salida:** entidades clínicas estructuradas por nota.
- **Decisión:** apoyo a la conciliación de medicamentos, con revisión profesional.

## Requisitos Previos

**Instalación** (una sola vez, desde la terminal; en Jupyter o Colab antepón `!`):

```bash
pip install pandas numpy scikit-learn torch transformers seqeval
```

**Librerías que se importan:**

| Librería | Para qué se usa |
|---------|-----------------|
| `os` | Rutas y creación de carpetas |
| `pandas, numpy` | Carga y preparación de los datos |
| `torch, torch.utils.data` | Tensores, `Dataset`, `DataLoader` y entrenamiento |
| `transformers` | `BertTokenizerFast` y `BertForTokenClassification` |
| `sklearn.metrics.accuracy_score` | Exactitud por token durante el entrenamiento |
| `seqeval.metrics` | Precisión, *recall* y F1 por entidad |

**Datos:** `data/ner_datasetreference.csv` (corpus “Annotated Corpus for NER” de Kaggle, ~1 millón de palabras en inglés). **Modelo:** `bert-base-uncased` (~420 MB) se descarga de Hugging Face la primera vez. Sin GPU, entrenar con todo el corpus tarda horas: usa `FRACCION_DATOS` (sección de preparación) para trabajar con una muestra.

## Marco metodológico: CRISP-ML(Q)

Este notebook sigue **CRISP-ML(Q)** (*Cross-Industry Standard Process for the development of Machine Learning applications with Quality assurance*; Studer et al., 2021). El proceso tiene seis fases, y en cada una se identifican los riesgos de calidad y cómo se mitigan. Las fases aparecen marcadas con 🔷 a lo largo del notebook.

| Fase CRISP-ML(Q) | Dónde se aborda en este notebook | Aseguramiento de calidad |
|------------------|----------------------------------|--------------------------|
| 1. Entendimiento del negocio y de los datos | Descarga y exploración: oraciones, palabras y frecuencia de etiquetas | Se descartan tipos de entidad con muy pocos ejemplos |
| 2. Ingeniería de datos | Agrupación por oración, `Dataset` con alineación de etiquetas a subpalabras, partición 80/20 y `DataLoader` | Se verifica a mano un ejemplo tokenizado y sus etiquetas |
| 3. Ingeniería del modelo | Modelo `BertForTokenClassification`, prueba de cordura de la pérdida inicial y entrenamiento | La pérdida inicial debe ser ≈ ln(número de etiquetas) |
| 4. Aseguramiento de calidad del modelo | Evaluación con exactitud por token y F1 por entidad (`seqeval`); inferencia en frases nuevas | Se explica por qué la exactitud por token engaña (domina la etiqueta O) |
| 5. Despliegue | Guardado del modelo y del tokenizador | Se recargan con `from_pretrained()` |
| 6. Monitoreo y mantenimiento | Al final (fuera de alcance) | — |

## Estándares de referencia

Prácticas de este notebook alineadas con estándares de IEEE SA, ISO/IEC y la Open Source Initiative (OSI). Se citan como guía de buenas prácticas, no como certificación de cumplimiento.

| Estándar | Cómo se aplica aquí |
|----------|---------------------|
| **IEEE 3119-2025** — Adquisición (*procurement*) de IA y sistemas de decisión automatizada | BERT es un modelo de terceros: se registra su origen (Google, `bert-base-uncased`), su licencia (Apache 2.0) y cómo se evaluó tras el ajuste. |
| **OSI · Open Source AI Definition 1.0 (2024)** — Qué hace que un sistema de IA sea de código abierto: libertades de usar, estudiar, modificar y compartir, con acceso a código, parámetros e información de los datos | BERT publica pesos, código y descripción de sus datos de preentrenamiento; el modelo ajustado puede compartirse con la misma apertura. |
| **IEEE 7003-2024** — Consideraciones sobre sesgo algorítmico | Se reporta F1 **por tipo de entidad**: un buen promedio puede ocultar entidades poco frecuentes mal reconocidas. |
| **IEEE 7001-2021** — Transparencia de sistemas autónomos | Se documentan las decisiones de diseño (alineación con la primera subpalabra, tipos de entidad descartados) y sus consecuencias. |
| **ISO/IEC 23894:2023** — Guía para la gestión de riesgos de IA | Riesgos tratados: métrica engañosa (exactitud dominada por O), costo de entrenamiento en CPU y errores de alineación entre palabras y subpalabras. |

## Objetivos de aprendizaje

1. Entender el esquema de anotación **BIO** usado en los conjuntos de datos de NER
2. Preparar un conjunto de datos de NER a nivel de token para ajustar BERT
3. Ajustar `bert-base-uncased` con `BertForTokenClassification`
4. Evaluar con F1 **por entidad** usando `seqeval`
5. Hacer inferencia sobre frases nuevas y guardar el modelo

## Ajuste fino de BERT para reconocimiento de entidades nombradas

Usaremos **`BertForTokenClassification`** de la librería [Transformers](https://github.com/huggingface/transformers) de Hugging Face: BERT como arquitectura base con una capa de clasificación por token encima, de modo que el modelo predice una etiqueta **para cada token** y no una sola para toda la secuencia. NER se suele plantear justamente como clasificación de tokens.

Se aplica **aprendizaje por transferencia**: BERT se preentrenó sin supervisión en dos tareas (modelado de lenguaje enmascarado y predicción de la siguiente oración) y ahora lo **ajustamos** con un conjunto etiquetado de NER, lo cual sí es aprendizaje supervisado.

Para saber más de BERT: el [artículo original](https://arxiv.org/abs/1810.04805) y la guía visual de Jay Alammar ([blog](http://jalammar.github.io/illustrated-bert/)).

Este notebook sigue la estructura de los tutoriales de Abhishek Kumar Mishra sobre Transformers ([repositorio](https://github.com/abhimishra91/transformers-tutorials)). Supone conocimientos básicos de aprendizaje profundo, BERT y PyTorch.

### Importar librerías y preparar el entorno

Las librerías necesarias están en **Requisitos previos**. En Google Colab, `transformers` y `seqeval` se instalan con la celda siguiente.

In [ ]:
import os
import sys
# Carpeta de datos relativa al notebook
PATH = os.getcwd()
DIR_DATA = os.path.join(PATH, 'data')
if PATH not in sys.path:
    sys.path.append(PATH)

In [ ]:
# En Colab: !pip install transformers seqeval

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizerFast, BertForTokenClassification

MODELO = 'bert-base-uncased'   # modelo preentrenado de Hugging Face

El aprendizaje profundo se acelera mucho con una GPU. En Google Colab: *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU*.

La siguiente celda elige la GPU si está disponible (imprime `cuda`) y, si no, la CPU.

In [ ]:
from torch import cuda
# GPU si está disponible; si no, CPU
device = 'cuda' if cuda.is_available() else 'cpu'
print(device)

> 🔷 **CRISP-ML(Q) · Fase 1 — Entendimiento del negocio y de los datos.** Corpus BIO, número de oraciones y frecuencia de cada tipo de entidad.

### Descargar y preprocesar los datos

NER usa un esquema de anotación definido a nivel de **palabra**. Uno muy común es **[IOB / BIO](https://en.wikipedia.org/wiki/Inside%E2%80%93outside%E2%80%93beginning_(tagging))** (*Inside-Outside-Beginning*): cada etiqueta indica si la palabra está al **inicio** (B), **dentro** (I) o **fuera** (O) de una entidad. Se usa porque muchas entidades tienen más de una palabra.

Ejemplo: en *“Barack Obama was born in Hawaii”* las etiquetas serían `[B-per, I-per, O, O, O, B-geo]`: *Barack* inicia una persona, *Obama* continúa esa persona, *was* está fuera de toda entidad, y así sucesivamente. Hay una etiqueta por palabra.

Para entrenar un modelo de NER hacen falta datos en este formato (o en variantes como BILOU). Herramientas como [Doccano](https://github.com/doccano/doccano) o Prodigy permiten anotarlos.

Usamos el corpus de NER de [Kaggle](https://www.kaggle.com/namanj27/ner-dataset), que ya viene en formato BIO, guardado en `data/ner_datasetreference.csv`. Veamos las primeras filas:

In [ ]:
# Corpus de NER de Kaggle: una fila por palabra; 'Sentence #' solo aparece en la primera palabra de cada oración
data = pd.read_csv(os.path.join(DIR_DATA, 'ner_datasetreference.csv'), encoding='unicode_escape')
data.head(20)

Contemos cuántas oraciones y palabras (con sus etiquetas) hay:

In [ ]:
# Valores no nulos por columna: 'Sentence #' cuenta oraciones; 'Word' y 'Tag', palabras
data.count()

Hay unas 48 000 oraciones y más de 1 millón de palabras y etiquetas, unas 20 palabras por oración.

Veamos las etiquetas de NER y su frecuencia:

In [ ]:
# Número de etiquetas distintas y frecuencia de cada una
print(f'Número de etiquetas: {data.Tag.nunique()}')
frequencies = data.Tag.value_counts()
frequencies

Hay 8 tipos de entidad, cada uno con variante de inicio (B) e interior (I), más la etiqueta O: 17 en total. Los tipos son:

| Tipo | Significado |
|------|-------------|
| `geo` | Entidad geográfica |
| `gpe` | Entidad geopolítica (países, ciudades como entidades políticas) |
| `org` | Organización |
| `per` | Persona |
| `tim` | Expresión de tiempo |
| `art` | Artefacto (obras, productos) |
| `eve` | Evento |
| `nat` | Fenómeno natural |

Algunos tipos están muy poco representados. Sumemos B e I por tipo:

In [ ]:
# Suma las ocurrencias B- e I- de cada tipo de entidad (los 3 caracteres después de "B-"/"I-")
tags = {}
for tag, count in zip(frequencies.index, frequencies):
    if tag != "O":
        tags[tag[2:5]] = tags.get(tag[2:5], 0) + count

print(sorted(tags.items(), key=lambda x: x[1], reverse=True))

Eliminamos las entidades `art`, `eve` y `nat`: tienen tan pocos ejemplos que su desempeño no sería comparable con el de las demás.

In [ ]:
# Se descartan las palabras etiquetadas con tipos poco frecuentes
entities_to_remove = ["B-art", "I-art", "B-eve", "I-eve", "B-nat", "I-nat"]
data = data[~data.Tag.isin(entities_to_remove)]
data.head()

> 🔷 **CRISP-ML(Q) · Fase 2 — Ingeniería de datos.** Oraciones, alineación de etiquetas con subpalabras, partición y DataLoaders.

Creamos dos diccionarios: etiqueta → índice e índice → etiqueta. El modelo trabaja con números, no con texto.

In [ ]:
# Diccionarios de etiquetas (el orden es el de aparición en el corpus)
labels_to_ids = {k: v for v, k in enumerate(data.Tag.unique())}
ids_to_labels = {v: k for v, k in enumerate(data.Tag.unique())}
print(f'Etiquetas tras el filtro: {len(labels_to_ids)}')
labels_to_ids

Quedan **11 etiquetas**: 5 tipos de entidad × (B, I) + O.

¿Qué es un ejemplo de entrenamiento en NER? Normalmente una **oración** con sus etiquetas BIO. Agrupemos las palabras y etiquetas por oración:

In [ ]:
# 'Sentence #' solo está en la primera palabra de cada oración: se rellena hacia abajo con el último valor no nulo
data = data.ffill()
data.head()

In [ ]:
# Columna "sentence": las palabras de cada oración unidas por espacios
data['sentence'] = data[['Sentence #', 'Word', 'Tag']].groupby(['Sentence #'])['Word'].transform(lambda x: ' '.join(x))
# Columna "word_labels": las etiquetas de cada oración unidas por comas
data['word_labels'] = data[['Sentence #', 'Word', 'Tag']].groupby(['Sentence #'])['Tag'].transform(lambda x: ','.join(x))
data.head()

Conservamos solo las columnas `sentence` y `word_labels` y eliminamos duplicados (cada oración aparecía una vez por palabra):

In [ ]:
# Una fila por oración
data = data[['sentence', 'word_labels']].drop_duplicates().reset_index(drop=True)
data.head()

In [ ]:
# Número de oraciones
len(data)

Verifiquemos que una oración al azar y sus etiquetas coincidan:

In [ ]:
# Oración de ejemplo
data.iloc[41].sentence

In [ ]:
# Sus etiquetas, una por palabra
data.iloc[41].word_labels

### Preparar el conjunto de datos y el `DataLoader`

Con los datos preprocesados, los convertimos en tensores de PyTorch. Primero definimos las variables clave del entrenamiento:

In [ ]:
MAX_LEN = 128            # longitud máxima en subpalabras por oración
TRAIN_BATCH_SIZE = 4
VALID_BATCH_SIZE = 2
EPOCHS = 1
LEARNING_RATE = 1e-05
MAX_GRAD_NORM = 10       # recorte de la norma del gradiente
# Fracción de oraciones a usar: 1.0 = todo el corpus. Sin GPU, una muestra (p. ej. 0.05) evita horas de entrenamiento
FRACCION_DATOS = 1.0 if device == 'cuda' else 0.05

tokenizer = BertTokenizerFast.from_pretrained(MODELO)

Un punto delicado del NER con BERT es que BERT tokeniza en **subpalabras (WordPiece)**, no en palabras, así que las etiquetas también deben definirse a nivel de subpalabra.

Por ejemplo, si *“Washington”* tiene la etiqueta `B-gpe` y se divide en *“wash”*, *“##ing”*, *“##ton”*, una opción es entrenar solo con la etiqueta de la **primera subpalabra** (solo *“wash”* recibe `B-gpe`), como en el artículo original de BERT. Es una **decisión de diseño**: también se puede propagar la etiqueta de la palabra a todas sus subpalabras.

La clase `Dataset` de abajo tokeniza cada oración, agrega los tokens especiales de BERT, rellena o trunca hasta `MAX_LEN`, crea la máscara de atención y asigna las etiquetas. Usamos `word_ids()` del tokenizador rápido, que indica a qué palabra pertenece cada subpalabra: la primera subpalabra de cada palabra recibe su etiqueta, y las demás (y los tokens especiales y de relleno) reciben **-100**, el `ignore_index` por defecto de [`CrossEntropyLoss`](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html), así que no cuentan en la pérdida.

In [ ]:
class dataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len):
        self.len = len(dataframe)
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __getitem__(self, index):
        # Paso 1: palabras y etiquetas de la oración
        sentence = self.data.sentence[index].strip().split()
        word_labels = self.data.word_labels[index].split(",")

        # Paso 2: tokenización en subpalabras (con relleno y truncado a max_len)
        encoding = self.tokenizer(sentence,
                                  is_split_into_words=True,
                                  padding='max_length',
                                  truncation=True,
                                  max_length=self.max_len)

        # Paso 3: etiqueta solo en la primera subpalabra de cada palabra; -100 en el resto
        encoded_labels = []
        palabra_anterior = None
        for word_id in encoding.word_ids():
            if word_id is None:                    # [CLS], [SEP] o relleno
                encoded_labels.append(-100)
            elif word_id != palabra_anterior:      # primera subpalabra de una palabra
                encoded_labels.append(labels_to_ids[word_labels[word_id]])
            else:                                  # subpalabras siguientes
                encoded_labels.append(-100)
            palabra_anterior = word_id

        # Paso 4: todo a tensores de PyTorch
        item = {key: torch.as_tensor(val) for key, val in encoding.items()}
        item['labels'] = torch.as_tensor(encoded_labels)
        return item

    def __len__(self):
        return self.len

Con esta clase creamos dos conjuntos, entrenamiento y prueba, con una partición 80/20:

In [ ]:
# Muestra opcional (FRACCION_DATOS) y partición 80/20 reproducible
data_uso = data.sample(frac=FRACCION_DATOS, random_state=200).reset_index(drop=True)
train_size = 0.8
train_dataset = data_uso.sample(frac=train_size, random_state=200)
test_dataset = data_uso.drop(train_dataset.index).reset_index(drop=True)
train_dataset = train_dataset.reset_index(drop=True)

print(f"Conjunto usado: {data_uso.shape} (fracción {FRACCION_DATOS})")
print(f"Entrenamiento:  {train_dataset.shape}")
print(f"Prueba:         {test_dataset.shape}")

training_set = dataset(train_dataset, tokenizer, MAX_LEN)
testing_set = dataset(test_dataset, tokenizer, MAX_LEN)

Veamos el primer ejemplo de entrenamiento:

In [ ]:
# Diccionario con input_ids, token_type_ids, attention_mask y labels
training_set[0]

Verifiquemos que cada subpalabra tenga la etiqueta correcta (-100 = ignorada):

In [ ]:
# Subpalabra y etiqueta asignada, hasta el primer token de relleno
for token, label in zip(tokenizer.convert_ids_to_tokens(training_set[0]["input_ids"]), training_set[0]["labels"]):
    if token == '[PAD]':
        break
    print('{0:15}  {1}'.format(token, ids_to_labels.get(int(label), int(label))))

Definimos los `DataLoader` de PyTorch:

In [ ]:
train_params = {'batch_size': TRAIN_BATCH_SIZE, 'shuffle': True, 'num_workers': 0}
test_params = {'batch_size': VALID_BATCH_SIZE, 'shuffle': False, 'num_workers': 0}   # prueba en orden fijo

training_loader = DataLoader(training_set, **train_params)
testing_loader = DataLoader(testing_set, **test_params)

> 🔷 **CRISP-ML(Q) · Fase 3 — Ingeniería del modelo.** BERT para clasificación de tokens, prueba de cordura y entrenamiento.

### Definir el modelo

Cargamos `BertForTokenClassification` con los pesos preentrenados de `bert-base-uncased`. Solo hay que indicar el número de etiquetas, que define el tamaño de la capa de clasificación.

Solo las capas base vienen preentrenadas: la capa de clasificación por token se inicializa al azar y se entrena junto con el resto. Por eso la librería muestra un aviso al ejecutar la celda.

In [ ]:
# BERT preentrenado + capa de clasificación nueva con una salida por etiqueta
model = BertForTokenClassification.from_pretrained(MODELO, num_labels=len(labels_to_ids))
model.to(device)

### Entrenar el modelo

Antes de entrenar hacemos una **prueba de cordura** (recomendada por Andrej Karpathy en [cs231n](http://cs231n.stanford.edu/) y en su [guía para depurar redes](http://karpathy.github.io/2019/04/25/recipe/)): la pérdida inicial debe ser cercana a $-\ln(1/K) = \ln K$, con $K$ = número de etiquetas. Con 11 etiquetas, $\ln 11 \approx 2.40$.

¿Por qué? La entropía cruzada es $-\ln(\text{probabilidad de la clase correcta})$. Al inicio la capa de clasificación es aleatoria, así que la probabilidad de cada etiqueta es cercana a $1/K$. `CrossEntropyLoss` promedia la pérdida sobre los tokens con etiqueta (los -100 no cuentan).

In [ ]:
# Pérdida inicial con un solo ejemplo, sin entrenar
inputs = training_set[2]
input_ids = inputs["input_ids"].unsqueeze(0).to(device)
attention_mask = inputs["attention_mask"].unsqueeze(0).to(device)
labels = inputs["labels"].unsqueeze(0).to(device)

outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
initial_loss = outputs.loss
print(f'Pérdida inicial: {initial_loss.item():.3f} · esperada ≈ ln({len(labels_to_ids)}) = {np.log(len(labels_to_ids)):.3f}')

Comprobemos también que los *logits* tengan forma `(batch_size, longitud_de_secuencia, número_de_etiquetas)`:

In [ ]:
tr_logits = outputs.logits
tr_logits.shape

Definimos el optimizador: Adam con tasa de aprendizaje fija. Se podrían usar AdamW (con decaimiento de pesos) y un programador de tasa de aprendizaje, pero aquí mantenemos lo simple.

In [ ]:
optimizer = torch.optim.Adam(params=model.parameters(), lr=LEARNING_RATE)

Ahora la función de entrenamiento en PyTorch, basada en parte en [este repositorio de NER multilingüe](https://github.com/chambliss/Multilingual_NER).

In [ ]:
# Una época de entrenamiento sobre el 80 % de los datos
def train(epoch):
    tr_loss, tr_accuracy = 0, 0
    nb_tr_examples, nb_tr_steps = 0, 0

    model.train()

    for idx, batch in enumerate(training_loader):
        ids = batch['input_ids'].to(device, dtype=torch.long)
        mask = batch['attention_mask'].to(device, dtype=torch.long)
        labels = batch['labels'].to(device, dtype=torch.long)

        # Los modelos de Hugging Face devuelven un ModelOutput con .loss y .logits
        outputs = model(input_ids=ids, attention_mask=mask, labels=labels)
        loss = outputs.loss
        tr_logits = outputs.logits

        tr_loss += loss.item()
        nb_tr_steps += 1
        nb_tr_examples += labels.size(0)

        if idx % 100 == 0:
            print(f"Pérdida de entrenamiento (promedio hasta el paso {idx}): {tr_loss / nb_tr_steps:.4f}")

        # Exactitud por token, solo en posiciones con etiqueta (≠ -100)
        flattened_targets = labels.view(-1)
        flattened_predictions = torch.argmax(tr_logits.view(-1, model.num_labels), axis=1)
        active = labels.view(-1) != -100
        tr_accuracy += accuracy_score(torch.masked_select(flattened_targets, active).cpu().numpy(),
                                      torch.masked_select(flattened_predictions, active).cpu().numpy())

        # Orden correcto: borrar gradientes → retropropagar → recortar → actualizar
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(parameters=model.parameters(), max_norm=MAX_GRAD_NORM)
        optimizer.step()

    print(f"Pérdida de entrenamiento de la época: {tr_loss / nb_tr_steps:.4f}")
    print(f"Exactitud de entrenamiento de la época: {tr_accuracy / nb_tr_steps:.4f}")

¡A entrenar!

In [ ]:
for epoch in range(EPOCHS):
    print(f"Época de entrenamiento: {epoch + 1}")
    train(epoch)

> 🔷 **CRISP-ML(Q) · Fase 4 — Aseguramiento de calidad del modelo.** Exactitud por token, F1 por entidad e inferencia en frases nuevas.

### Evaluar el modelo

Con el modelo entrenado, medimos su desempeño en el 20 % reservado. No hay actualización de pesos: el modelo solo produce *logits*. Guardamos las etiquetas **por oración**, porque `seqeval` necesita secuencias para reconstruir las entidades.

In [ ]:
def valid(model, testing_loader):
    # Modo evaluación: desactiva el dropout
    model.eval()

    eval_loss, eval_accuracy, nb_eval_steps = 0, 0, 0
    eval_labels, eval_preds = [], []      # una lista de etiquetas por oración

    with torch.no_grad():
        for idx, batch in enumerate(testing_loader):
            ids = batch['input_ids'].to(device, dtype=torch.long)
            mask = batch['attention_mask'].to(device, dtype=torch.long)
            labels = batch['labels'].to(device, dtype=torch.long)

            outputs = model(input_ids=ids, attention_mask=mask, labels=labels)
            eval_loss += outputs.loss.item()
            nb_eval_steps += 1

            if idx % 100 == 0:
                print(f"Pérdida de validación (promedio hasta el paso {idx}): {eval_loss / nb_eval_steps:.4f}")

            predictions = torch.argmax(outputs.logits, axis=-1)   # (batch, seq_len)
            # Por oración: solo posiciones con etiqueta real (primera subpalabra de cada palabra)
            for y_true, y_pred in zip(labels, predictions):
                activo = y_true != -100
                verdad = [ids_to_labels[i.item()] for i in y_true[activo]]
                pred = [ids_to_labels[i.item()] for i in y_pred[activo]]
                eval_labels.append(verdad)
                eval_preds.append(pred)
                eval_accuracy += accuracy_score(verdad, pred)

    print(f"Pérdida de validación: {eval_loss / nb_eval_steps:.4f}")
    print(f"Exactitud por token (promedio por oración): {eval_accuracy / len(eval_labels):.4f}")
    return eval_labels, eval_preds

Evaluemos en el conjunto de prueba:

In [ ]:
labels, predictions = valid(model, testing_loader)

La exactitud por token **engaña**: la mayoría de las palabras tienen la etiqueta O, así que un modelo que casi nunca detecta entidades puede tener una exactitud muy alta. Lo importante es la precisión, el *recall* y el F1 **por entidad**: una entidad cuenta como acierto solo si su tipo y sus límites coinciden exactamente. Para eso usamos `seqeval`:

In [ ]:
from seqeval.metrics import classification_report

# seqeval recibe listas de secuencias (una lista de etiquetas por oración)
print(classification_report(labels, predictions))

Revisa el F1 de cada tipo de entidad: los tipos con menos ejemplos suelen quedar por debajo. Con una sola época (y, sin GPU, con una muestra de los datos) el modelo aún puede mejorar; lo recomendable sería evaluar en un conjunto de validación durante el entrenamiento para elegir el número de épocas.

### Inferencia

Lo interesante es probar el modelo con frases nuevas. Usamos la predicción de la **primera subpalabra de cada palabra**, que es como se entrenó el modelo. El código no considera los casos en que las subpalabras de una misma palabra reciben predicciones distintas.

In [ ]:
sentence = "@HuggingFace is a company based in New York, but it also has employees working in Paris"

inputs = tokenizer(sentence.split(),
                   is_split_into_words=True,
                   padding='max_length',
                   truncation=True,
                   max_length=MAX_LEN,
                   return_tensors="pt")

# Paso hacia adelante sin gradientes
model.eval()
with torch.no_grad():
    logits = model(inputs["input_ids"].to(device), attention_mask=inputs["attention_mask"].to(device)).logits
flattened_predictions = torch.argmax(logits.view(-1, model.num_labels), axis=1).cpu().numpy()

# Solo la predicción de la primera subpalabra de cada palabra
prediction, palabra_anterior = [], None
for posicion, word_id in enumerate(inputs.word_ids()):
    if word_id is not None and word_id != palabra_anterior:
        prediction.append(ids_to_labels[flattened_predictions[posicion]])
    palabra_anterior = word_id

for palabra, etiqueta in zip(sentence.split(), prediction):
    print(f'{palabra:12s} {etiqueta}')

> 🔷 **CRISP-ML(Q) · Fase 5 — Despliegue.** Guardar modelo y tokenizador para recargarlos con `from_pretrained()`.

### Guardar el modelo para usarlo después

Guardamos el tokenizador, los pesos del modelo y su configuración en una carpeta, para recargarlos con `from_pretrained()`.

In [ ]:
directory = "./model"
os.makedirs(directory, exist_ok=True)

# Tokenizador completo (vocabulario y configuración)
tokenizer.save_pretrained(directory)
# Pesos del modelo y archivo de configuración
model.save_pretrained(directory)
print(f'Archivos guardados en {directory}')

---

> 🔷 **CRISP-ML(Q) · Fase 6 — Monitoreo y mantenimiento** (fuera del alcance de este notebook): registrar la proporción de entidades detectadas por tipo y revisar una muestra de predicciones cada cierto tiempo. Si aparecen nuevos tipos de entidad o cambia el estilo de los textos, anotar ejemplos nuevos y repetir el ajuste.